Notebook 2 — 02_Leakage_Safe_Preprocessing.ipynb
Create a new notebook with this name in your existing Notebooks 2 folder. We will audit the existing pipeline first, rather than immediately retraining the models.

In [1]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

# Locate the repository root.
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (
        (PROJECT_ROOT / "data" / "processed").is_dir()
        and (PROJECT_ROOT / "models").is_dir()
        and (PROJECT_ROOT / "notebooks").is_dir()
    ):
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "notebooks").is_dir(), (
    "Project root not found. Set PROJECT_ROOT to your repository."
)

print("Project root:", PROJECT_ROOT)

AUDIT_DIR = PROJECT_ROOT / "reports" / "research_audit" / "common_patient_split"
CNN_DIR = PROJECT_ROOT / "data" / "processed" / "deep_learning" / "cnn_baseline"

print("Research audit directory exists:", AUDIT_DIR.exists())
print("CNN experiment directory exists:", CNN_DIR.exists())


Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification
Research audit directory exists: True
CNN experiment directory exists: True


In [2]:

files_to_check = [
    PROJECT_ROOT / "data/processed/feature_selection/selected_features.csv",
    PROJECT_ROOT / "data/processed/classical_ml/classical_ml_selected_features.csv",
    PROJECT_ROOT / "reports/research_audit/common_patient_split/classical_selected_features.json",
    PROJECT_ROOT / "data/processed/deep_learning/logmel_train_normalization.csv",
    PROJECT_ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_train_normalization.csv",
    PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline/cnn_training_config.json",
    PROJECT_ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_training_config.json",
]

for path in files_to_check:
    print("\n" + "=" * 70)
    print(path.relative_to(PROJECT_ROOT))

    if not path.exists():
        print("Not found")
        continue

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(10))

    elif path.suffix.lower() == ".json":
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)
        print(json.dumps(obj, indent=2)[:10000])



data/processed/feature_selection/selected_features.csv
Shape: (33, 1)
Columns: ['feature']


,feature
0,spectral_entropy
1,mfcc_11_mean
2,zero_crossing_rate
3,dominant_frequency
4,mfcc_13_mean
5,wavelet_entropy
6,mfcc_1_mean
7,spectral_bandwidth
8,spectral_rolloff
9,mfcc_5_mean



data/processed/classical_ml/classical_ml_selected_features.csv
Shape: (33, 1)
Columns: ['feature']


,feature
0,spectral_entropy
1,mfcc_11_mean
2,zero_crossing_rate
3,dominant_frequency
4,mfcc_13_mean
5,wavelet_entropy
6,mfcc_1_mean
7,spectral_bandwidth
8,spectral_rolloff
9,mfcc_5_mean



reports/research_audit/common_patient_split/classical_selected_features.json
[
  "spectral_entropy",
  "mfcc_11_mean",
  "zero_crossing_rate",
  "wavelet_entropy",
  "mfcc_3_mean",
  "dominant_frequency",
  "spectral_centroid",
  "mfcc_13_mean",
  "mfcc_1_mean",
  "spectral_bandwidth",
  "mfcc_7_mean",
  "mfcc_4_mean",
  "mfcc_5_mean",
  "mfcc_9_mean",
  "energy",
  "mfcc_2_mean",
  "mfcc_12_mean",
  "band_energy_500_1000",
  "mfcc_10_mean",
  "spectral_flatness",
  "band_energy_0_500",
  "wavelet_std",
  "peak_to_peak",
  "band_energy_1000_1500",
  "min",
  "mfcc_8_mean",
  "median",
  "kurtosis",
  "mfcc_6_mean",
  "skewness",
  "mean",
  "max",
  "band_energy_1500_2000"
]

data/processed/deep_learning/logmel_train_normalization.csv
Shape: (2, 2)
Columns: ['statistic', 'value']


,statistic,value
0,mean,-66.740814
1,std,17.337090



data/processed/deep_learning/multifeature_cnn/multifeature_train_normalization.csv
Shape: (3, 4)
Columns: ['feature', 'count', 'mean', 'std']


,feature,count,mean,std
0,logmel,44442304,-66.230481,17.661886
1,mfcc,9027343,-22.509034,137.274416
2,chroma,8332932,0.385656,0.389059



data/processed/deep_learning/cnn_baseline/cnn_training_config.json
{
  "batch_size": 32,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "max_epochs": 40,
  "early_stopping_patience": 7,
  "lr_reduction_factor": 0.5,
  "lr_scheduler_patience": 3,
  "minimum_learning_rate": 1e-06,
  "optimizer": "Adam",
  "loss": "Weighted CrossEntropyLoss",
  "augmentation": [
    "Gaussian noise",
    "random time shift"
  ],
  "augmentation_training_only": true,
  "validation_monitor": "loss"
}

data/processed/deep_learning/multifeature_cnn/multifeature_cnn_training_config.json
{
  "batch_size": 32,
  "max_epochs": 40,
  "early_stopping_patience": 7,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "scheduler": "ReduceLROnPlateau",
  "scheduler_factor": 0.5,
  "scheduler_patience": 3,
  "scheduler_min_lr": 1e-06,
  "loss": "weighted_cross_entropy",
  "optimizer": "Adam",
  "augmentation": [
    "random_time_shift",
    "gaussian_noise"
  ],
  "feature_inputs": [
    "Log-Mel",
    "MFCC",

In [3]:

import nbformat

notebooks_to_audit = [
    "07_feature_selection.ipynb",
    "09_classical_ml.ipynb",
    "10_deep_learning_data_preparation.ipynb",
    "11_cnn_baseline.ipynb",
    "12_multifeature_cnn.ipynb",
    "14_patient_level_prediction.ipynb",
]

keywords = [
    "fit_transform",
    ".fit(",
    "StandardScaler",
    "MinMaxScaler",
    "SelectKBest",
    "feature_selection",
    "train_test_split",
    "GroupShuffleSplit",
    "GroupKFold",
    "StratifiedGroupKFold",
    "class_weight",
    "WeightedRandomSampler",
    "normalization",
    "mean(",
    "std(",
    "SMOTE",
    "resample",
]

for notebook_name in notebooks_to_audit:
    path = PROJECT_ROOT / "notebooks" / notebook_name

    print("\n" + "=" * 80)
    print("NOTEBOOK:", notebook_name)

    if not path.exists():
        print("Notebook not found at:", path)
        continue

    notebook = nbformat.read(path, as_version=4)

    for cell_number, cell in enumerate(notebook.cells, start=1):
        if cell.cell_type != "code":
            continue

        source = cell.source
        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            print(f"\nCell {cell_number} | Matches: {matched}")
            print(source[:1800])
            if len(source) > 1800:
                print("... [truncated]")



NOTEBOOK: 07_feature_selection.ipynb

Cell 2 | Matches: ['feature_selection']
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_selection import (
    f_classif,
    mutual_info_classif
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import LabelEncoder

print("Libraries imported successfully.")

Cell 11 | Matches: ['fit_transform']
# ============================================================
# Prepare Training Data for Feature Selection
# ============================================================

train_df = features_with_split[
    features_with_split["official_split"] == "train"
].copy()

test_df = features_with_split[
    features_with_split["official_split"] == "test"
].copy()

X_train = train_df[feature_columns].copy()

label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(
    train_df["diagnosis"]
)

print("Training cycles:", len(train_df))
print("Testing

In [4]:

import nbformat
import pandas as pd

# ---------------------------------------------------------
# 1. Compare the two saved 33-feature lists
# ---------------------------------------------------------
general_path = (
    PROJECT_ROOT / "data/processed/feature_selection/selected_features.csv"
)
audit_path = (
    PROJECT_ROOT
    / "reports/research_audit/common_patient_split/classical_selected_features.json"
)

general_features = pd.read_csv(general_path)["feature"].tolist()

with open(audit_path, "r", encoding="utf-8") as f:
    audit_features = json.load(f)

print("GENERAL FEATURE LIST:", len(general_features))
print("CANONICAL AUDIT LIST:", len(audit_features))
print("Features shared:", len(set(general_features) & set(audit_features)))
print("Only in general list:", sorted(set(general_features) - set(audit_features)))
print("Only in canonical list:", sorted(set(audit_features) - set(general_features)))

# ---------------------------------------------------------
# 2. Compare official-split patients with canonical patients
# ---------------------------------------------------------
recordings_path = PROJECT_ROOT / "data/interim/recordings.csv"
recordings = pd.read_csv(recordings_path)

canonical_train = pd.read_csv(
    PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline/train_cnn_manifest.csv"
)
canonical_validation = pd.read_csv(
    PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline/validation_cnn_manifest.csv"
)
canonical_test = pd.read_csv(
    PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline/test_cnn_manifest.csv"
)

canonical_sets = {
    "train": set(canonical_train["patient_id"].unique()),
    "validation": set(canonical_validation["patient_id"].unique()),
    "test": set(canonical_test["patient_id"].unique()),
}

official_sets = {
    split: set(
        recordings.loc[
            recordings["official_split"].eq(split), "patient_id"
        ].unique()
    )
    for split in recordings["official_split"].dropna().unique()
}

print("\nOFFICIAL SPLIT PATIENT COUNTS")
for name, ids in official_sets.items():
    print(name, len(ids))

print("\nCANONICAL SPLIT PATIENT COUNTS")
for name, ids in canonical_sets.items():
    print(name, len(ids))

print("\nOVERLAP WITH OFFICIAL SPLITS")
for canonical_name, canonical_ids in canonical_sets.items():
    for official_name, official_ids in official_sets.items():
        print(
            f"Canonical {canonical_name} vs official {official_name}: "
            f"{len(canonical_ids & official_ids)} shared patients"
        )

# ---------------------------------------------------------
# 3. Show the exact split-building code in Notebook 07
# ---------------------------------------------------------
notebook_path = PROJECT_ROOT / "notebooks/07_feature_selection.ipynb"
nb = nbformat.read(notebook_path, as_version=4)

print("\nNOTEBOOK 07: SPLIT AND FEATURE DATA DEFINITIONS")
search_terms = [
    "features_with_split",
    "official_split",
    "feature_columns",
    "train_df =",
    "test_df =",
]

for i, cell in enumerate(nb.cells, start=1):
    if cell.cell_type != "code":
        continue

    source = cell.source
    if any(term in source for term in search_terms):
        print(f"\n--- Cell {i} ---")
        print(source[:3500])


GENERAL FEATURE LIST: 33
CANONICAL AUDIT LIST: 33
Features shared: 31
Only in general list: ['rms', 'spectral_rolloff']
Only in canonical list: ['spectral_centroid', 'wavelet_std']

OFFICIAL SPLIT PATIENT COUNTS
test 49
train 79

CANONICAL SPLIT PATIENT COUNTS
train 82
validation 19
test 25

OVERLAP WITH OFFICIAL SPLITS
Canonical train vs official test: 31 shared patients
Canonical train vs official train: 52 shared patients
Canonical validation vs official test: 7 shared patients
Canonical validation vs official train: 12 shared patients
Canonical test vs official test: 11 shared patients
Canonical test vs official train: 15 shared patients

NOTEBOOK 07: SPLIT AND FEATURE DATA DEFINITIONS

--- Cell 4 ---
metadata_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "cycle_number",
    "start_time",
    "end_time",
    "sound_label",
    "crackles",
    "wheezes",
    "diagnosis",
    "processed_path",
]

feature_columns =

In [5]:

# Cell 5 — Trace the canonical feature-selection artifacts
# Read-only audit: this cell does not modify project files.

import json
from pathlib import Path

audit_dir = (
    PROJECT_ROOT
    / "reports"
    / "research_audit"
    / "common_patient_split"
)

protocol_path = audit_dir / "protocol.json"
feature_path = audit_dir / "classical_selected_features.json"

print("========== CANONICAL FEATURE-SELECTION AUDIT ==========")

# 1. Inspect the documented protocol
if protocol_path.exists():
    with open(protocol_path, "r", encoding="utf-8") as f:
        protocol = json.load(f)

    print("\n--- Protocol details ---")
    for key in [
        "protocol",
        "feature_selection",
        "feature_count",
        "train_cycles",
        "validation_cycles",
        "test_cycles",
        "limitations",
    ]:
        print(f"\n{key}:")
        print(protocol.get(key, "<not recorded>"))
else:
    print("Protocol file not found:", protocol_path)

# 2. Inspect the canonical selected feature list
if feature_path.exists():
    with open(feature_path, "r", encoding="utf-8") as f:
        canonical_features = json.load(f)

    print("\n--- Canonical feature artifact ---")
    print("Artifact:", feature_path)
    print("Python object type:", type(canonical_features).__name__)

    if isinstance(canonical_features, dict):
        print("Top-level keys:", list(canonical_features.keys()))
        print(json.dumps(canonical_features, indent=2)[:12000])
    elif isinstance(canonical_features, list):
        print("Feature count:", len(canonical_features))
        for i, feature in enumerate(canonical_features, start=1):
            print(f"{i:02d}. {feature}")
    else:
        print(repr(canonical_features))
else:
    print("Canonical feature file not found:", feature_path)

# 3. Find relevant audit files without opening or modifying them
print("\n--- Related research-audit files ---")

if audit_dir.exists():
    for path in sorted(audit_dir.rglob("*")):
        if path.is_file() and any(
            term in path.name.lower()
            for term in [
                "feature",
                "classical",
                "train",
                "protocol",
                "audit",
                "experiment",
            ]
        ):
            print(path.relative_to(PROJECT_ROOT))
else:
    print("Audit directory not found:", audit_dir)


========== CANONICAL FEATURE-SELECTION AUDIT ==========

--- Protocol details ---

protocol:
cycle-level evaluation on the existing CNN patient-disjoint holdout

feature_selection:
Notebook 07 consensus ranking and 0.90 correlation pruning fitted on patient training data only

feature_count:
33

train_cycles:
4423

validation_cycles:
914

test_cycles:
1561

limitations:
['Cycle rows from a held-out patient are correlated; metrics are cycle-level, not independent patient-level estimates.', 'The patient test cohort has no Asthma or LRTI support; their per-class metrics are reported as unavailable.', 'Classical estimators were retrained with the saved experiment hyperparameters, not re-tuned on this split.', 'A complete audit of whether historical hyperparameter decisions were influenced by earlier test results remains necessary before claiming a final untouched benchmark.', 'Saved neural checkpoints are the existing patient-split models; they were not retrained in this script.', 'Inferen

In [6]:

# Cell 6 — Locate the canonical feature-selection implementation
# Read-only source search.

import json
from pathlib import Path

search_terms = [
    "classical_selected_features.json",
    "selection_corr_threshold",
    "train_correlation_matrix",
    "feature_ranking_df",
    "common_patient_split",
]

search_roots = [
    PROJECT_ROOT / "Notebooks",
    PROJECT_ROOT / "Notebooks 2",
    PROJECT_ROOT / "scripts",
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "reports",
]

extensions = {".py", ".ipynb"}
max_matches_per_term = 12
matches_by_term = {term: [] for term in search_terms}

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in extensions:
            continue

        # Skip generated environments and version-control internals.
        if any(
            part in {".git", ".venv", "venv", "__pycache__"}
            for part in path.parts
        ):
            continue

        try:
            if path.suffix.lower() == ".ipynb":
                notebook = json.loads(
                    path.read_text(encoding="utf-8")
                )
                chunks = []

                for cell_index, cell in enumerate(
                    notebook.get("cells", [])
                ):
                    source = cell.get("source", [])
                    if isinstance(source, list):
                        source = "".join(source)

                    chunks.append(
                        (cell_index, source)
                    )
            else:
                text = path.read_text(
                    encoding="utf-8",
                    errors="replace"
                )
                chunks = [
                    (None, text)
                ]

            for term in search_terms:
                if len(matches_by_term[term]) >= max_matches_per_term:
                    continue

                for cell_index, source in chunks:
                    if term.lower() not in source.lower():
                        continue

                    lines = source.splitlines()
                    matching_lines = [
                        i for i, line in enumerate(lines)
                        if term.lower() in line.lower()
                    ]

                    for line_index in matching_lines[:3]:
                        start = max(0, line_index - 5)
                        end = min(len(lines), line_index + 7)

                        matches_by_term[term].append({
                            "file": str(path.relative_to(PROJECT_ROOT)),
                            "cell": cell_index,
                            "line": line_index + 1,
                            "snippet": "\n".join(
                                f"{i + 1}: {lines[i]}"
                                for i in range(start, end)
                            ),
                        })

                        if len(matches_by_term[term]) >= max_matches_per_term:
                            break

                    if len(matches_by_term[term]) >= max_matches_per_term:
                        break

        except Exception as exc:
            print(f"Could not inspect {path}: {exc}")

print("========== CANONICAL SOURCE SEARCH ==========")

for term, matches in matches_by_term.items():
    print(f"\n--- Search term: {term} ---")

    if not matches:
        print("No matches found in the searched locations.")
        continue

    for match in matches:
        print(
            f"\nFile: {match['file']} | "
            f"Cell: {match['cell']} | "
            f"Line: {match['line']}"
        )
        print(match["snippet"])


========== CANONICAL SOURCE SEARCH ==========

--- Search term: classical_selected_features.json ---

File: Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb | Cell: 2 | Line: 5
1: 
2: files_to_check = [
3:     PROJECT_ROOT / "data/processed/feature_selection/selected_features.csv",
4:     PROJECT_ROOT / "data/processed/classical_ml/classical_ml_selected_features.csv",
5:     PROJECT_ROOT / "reports/research_audit/common_patient_split/classical_selected_features.json",
6:     PROJECT_ROOT / "data/processed/deep_learning/logmel_train_normalization.csv",
7:     PROJECT_ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_train_normalization.csv",
8:     PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline/cnn_training_config.json",
9:     PROJECT_ROOT / "data/processed/deep_learning/multifeature_cnn/multifeature_cnn_training_config.json",
10: ]
11: 

File: Notebooks 2/02_Leakage_Safe_Preprocessing.ipynb | Cell: 4 | Line: 13
8: general_path = (
9:     PROJECT_ROOT / "data

In [7]:

# Cell 7 — Inspect canonical benchmark implementation
# Read-only source inspection.

script_path = PROJECT_ROOT / "scripts" / "evaluate_common_patient_split.py"

if not script_path.exists():
    print("Script not found:", script_path)
else:
    source_lines = script_path.read_text(
        encoding="utf-8"
    ).splitlines()

    print("Script:", script_path)
    print("Total lines:", len(source_lines))

    sections = [
        ("Input paths and manifests", 1, 100),
        ("Feature selection and model fitting", 100, 280),
        ("Predictions, metrics and saved artifacts", 280, 430),
    ]

    for title, start, end in sections:
        print(f"\n{'=' * 15} {title} {'=' * 15}")

        for line_no in range(start, min(end, len(source_lines)) + 1):
            print(f"{line_no:04d}: {source_lines[line_no - 1]}")


Script: /Users/vs/Sem 5/Respiratory-Disease-Classification/scripts/evaluate_common_patient_split.py
Total lines: 427

=============== Input paths and manifests ===============
0001: """Retrain classical baselines and compare all models on the CNN patient holdout.
0002: 
0003: The script does not overwrite app artifacts or legacy experiment reports. It uses
0004: the CNN patient-disjoint manifests, fits feature selection/scaling/models on the
0005: training patients only, and checks row alignment for saved neural predictions.
0006: """
0007: 
0008: from __future__ import annotations
0009: 
0010: import json
0011: import pickle
0012: import time
0013: from pathlib import Path
0014: 
0015: import numpy as np
0016: import pandas as pd
0017: import joblib
0018: from sklearn.ensemble import RandomForestClassifier
0019: from sklearn.feature_selection import f_classif, mutual_info_classif
0020: from sklearn.linear_model import LogisticRegression
0021: from sklearn.metrics import (
0022:     ac

In [8]:

# Cell 8 — Independent validation of canonical ML inputs
# Read-only: no model training and no files are modified.

import json
import numpy as np
import pandas as pd

audit_dir = PROJECT_ROOT / "reports/research_audit/common_patient_split"
cnn_dir = PROJECT_ROOT / "data/processed/deep_learning/cnn_baseline"

features_path = (
    PROJECT_ROOT
    / "data/processed/features/respiratory_cycle_features.csv"
)

train_path = cnn_dir / "train_cnn_manifest.csv"
val_path = cnn_dir / "validation_cnn_manifest.csv"
test_path = cnn_dir / "test_cnn_manifest.csv"
selected_path = audit_dir / "classical_selected_features.json"

with open(selected_path, "r", encoding="utf-8") as f:
    selected_features = json.load(f)

features_df = pd.read_csv(features_path)
train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("========== CANONICAL INPUT VALIDATION ==========")

# 1. Feature availability
missing_features = [
    feature for feature in selected_features
    if feature not in features_df.columns
]

print("\n1. FEATURE AVAILABILITY")
print("Selected feature count:", len(selected_features))
print("Missing selected features:", missing_features)
print("Duplicate selected feature names:",
      len(selected_features) != len(set(selected_features)))

# 2. Patient separation
train_patients = set(train_df["patient_id"].unique())
val_patients = set(val_df["patient_id"].unique())
test_patients = set(test_df["patient_id"].unique())

print("\n2. PATIENT SEPARATION")
print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))
print("Train ∩ validation:", len(train_patients & val_patients))
print("Train ∩ test:", len(train_patients & test_patients))
print("Validation ∩ test:", len(val_patients & test_patients))

# 3. Manifest and feature-table row coverage
print("\n3. ROW COUNTS")
print("Feature-table rows:", len(features_df))
print("Train manifest rows:", len(train_df))
print("Validation manifest rows:", len(val_df))
print("Test manifest rows:", len(test_df))
print("Manifest total:", len(train_df) + len(val_df) + len(test_df))

# 4. Compare cycle identifiers
cycle_keys = [
    "patient_id",
    "recording_id",
    "chest_location",
    "cycle_number",
]

if all(col in features_df.columns for col in cycle_keys):
    print("\n4. CYCLE IDENTIFIERS")
    print("Duplicate feature-table cycle keys:",
          int(features_df.duplicated(cycle_keys).sum()))

    for split_name, manifest in [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df),
    ]:
        if all(col in manifest.columns for col in cycle_keys):
            feature_keys = pd.MultiIndex.from_frame(
                features_df[cycle_keys]
            )
            manifest_keys = pd.MultiIndex.from_frame(
                manifest[cycle_keys]
            )

            print(
                f"{split_name}: manifest rows without a matching "
                f"feature-table cycle = "
                f"{int((~manifest_keys.isin(feature_keys)).sum())}"
            )

# 5. Feature values: missing and non-finite values
print("\n5. FEATURE NUMERICS")

if not missing_features:
    numeric_features = features_df[selected_features].apply(
        pd.to_numeric, errors="coerce"
    )

    print("Rows with at least one missing/non-numeric selected value:",
          int(numeric_features.isna().any(axis=1).sum()))

    print("Total non-finite selected values:",
          int((~np.isfinite(numeric_features.to_numpy())).sum()))
else:
    print("Skipped numeric checks because selected columns are missing.")

# 6. Class imbalance on the canonical training partition
print("\n6. TRAINING CLASS DISTRIBUTION")

label_col = "disease_label"

if label_col in train_df.columns:
    train_counts = train_df[label_col].value_counts().sort_index()
    distribution = pd.DataFrame({
        "cycle_count": train_counts,
        "percentage": 100 * train_counts / len(train_df),
    })

    display(distribution.round(2))
    print("Largest/smallest represented class cycle ratio:",
          round(train_counts.max() / train_counts.min(), 2)
          if len(train_counts) and train_counts.min() > 0
          else "Undefined: at least one class has zero support")
else:
    print("No disease_label column found in training manifest.")

# 7. Confirm one diagnosis per patient in each split
print("\n7. PATIENT LABEL CONSISTENCY")

for split_name, manifest in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df),
]:
    if {"patient_id", label_col}.issubset(manifest.columns):
        labels_per_patient = manifest.groupby("patient_id")[label_col].nunique()
        inconsistent = int((labels_per_patient > 1).sum())

        print(
            f"{split_name}: patients with multiple labels = {inconsistent}"
        )

print("\nValidation cell completed. No files were modified.")


========== CANONICAL INPUT VALIDATION ==========

1. FEATURE AVAILABILITY
Selected feature count: 33
Missing selected features: []
Duplicate selected feature names: False

2. PATIENT SEPARATION
Train patients: 82
Validation patients: 19
Test patients: 25
Train ∩ validation: 0
Train ∩ test: 0
Validation ∩ test: 0

3. ROW COUNTS
Feature-table rows: 6898
Train manifest rows: 4423
Validation manifest rows: 914
Test manifest rows: 1561
Manifest total: 6898

4. CYCLE IDENTIFIERS
Duplicate feature-table cycle keys: 0

5. FEATURE NUMERICS
Rows with at least one missing/non-numeric selected value: 0
Total non-finite selected values: 0

6. TRAINING CLASS DISTRIBUTION


,cycle_count,percentage
disease_label,,
0,6,0.14
1,76,1.72
2,114,2.58
3,3713,83.95
4,194,4.39
5,32,0.72
6,121,2.74
7,167,3.78


Largest/smallest represented class cycle ratio: 618.83

7. PATIENT LABEL CONSISTENCY
train: patients with multiple labels = 0
validation: patients with multiple labels = 0
test: patients with multiple labels = 0

Validation cell completed. No files were modified.


In [9]:

# Cell 9 — Class imbalance by cycles, recordings and patients
# Read-only analysis of canonical manifests.

import pandas as pd

class_names = [
    "Asthma",
    "Bronchiectasis",
    "Bronchiolitis",
    "COPD",
    "Healthy",
    "LRTI",
    "Pneumonia",
    "URTI",
]

def class_support_table(manifest, split_name):
    rows = []

    for label, disease in enumerate(class_names):
        subset = manifest[manifest["disease_label"] == label]

        rows.append({
            "split": split_name,
            "label": label,
            "disease": disease,
            "cycles": len(subset),
            "recordings": (
                subset[["patient_id", "recording_id"]]
                .drop_duplicates()
                .shape[0]
            ),
            "patients": subset["patient_id"].nunique(),
        })

    return pd.DataFrame(rows)

support = pd.concat(
    [
        class_support_table(train_df, "train"),
        class_support_table(val_df, "validation"),
        class_support_table(test_df, "test"),
    ],
    ignore_index=True,
)

print("========== CLASS SUPPORT BY SPLIT ==========")
display(
    support.pivot(
        index="disease",
        columns="split",
        values=["cycles", "recordings", "patients"],
    ).fillna(0)
)

print("\n========== TRAINING PATIENT SUPPORT ==========")
display(
    support[support["split"] == "train"][
        ["disease", "cycles", "recordings", "patients"]
    ].sort_values("patients")
)

print(
    "\nNote: cycles and recordings from one patient are correlated; "
    "they are not independent patient observations."
)


========== CLASS SUPPORT BY SPLIT ==========


cycles                  recordings                  patients  \
split            test train validation       test train validation     test   
disease                                                                       
Asthma              0     6          0          0     1          0        0   
Bronchiectasis     20    76          8          3     8          1        2   
Bronchiolitis      19   114         27          1     4          1        1   
COPD             1337  3713        696         37   117         25       13   
Healthy            84   194         44          6    17          5        5   
LRTI                0    32          0          0     2          0        0   
Pneumonia          52   121        112          3     8          3        1   
URTI               49   167         27          3     9          2        3   

                                 
split          train validation  
disease                          
Asthma             1          0  
Bronchiectasis     4          1  
Bronchiolitis      4          1  
COPD              41         10  
Healthy           17          4  
LRTI               2          0  
Pneumonia          4          1  
URTI               9          2


========== TRAINING PATIENT SUPPORT ==========


,disease,cycles,recordings,patients
0,Asthma,6,1,1
5,LRTI,32,2,2
1,Bronchiectasis,76,8,4
2,Bronchiolitis,114,4,4
6,Pneumonia,121,8,4
7,URTI,167,9,9
4,Healthy,194,17,17
3,COPD,3713,117,41



Note: cycles and recordings from one patient are correlated; they are not independent patient observations.
